# Post-Selection Inference: Valid Confidence Intervals After Feature Selection

## Learning Objectives

By completing this notebook, you will:
1. Demonstrate empirically that naive OLS confidence intervals on Lasso-selected features have incorrect coverage
2. Implement the Debiased Lasso (van de Geer et al., 2014) and verify improved coverage
3. Implement data splitting for valid post-selection inference
4. Build a complete screen → select → infer pipeline incorporating all Module 8 methods

## Prerequisites
- Lasso and coordinate descent (Module 4)
- SIS screening (Notebook 01 of this module)
- Guide 03 of this module: *Post-Selection Inference*

## Estimated Time: 15 minutes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LassoCV, Lasso, LinearRegression
from sklearn.preprocessing import StandardScaler
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams.update({'font.size': 11, 'figure.dpi': 100})

## 1. Demonstrating the Post-Selection Bias Problem

We run a Monte Carlo simulation to measure the *actual* coverage of a "95%" confidence interval computed on Lasso-selected features using naive OLS. The expected coverage is 95% if the interval is valid — we will see it is far lower.

**Setup**: $n = 150$, $p = 300$, $s = 10$ active features. True model is linear with known coefficients.

In [ ]:
def generate_sparse_linear(
    n: int,
    p: int,
    s: int,
    snr: float = 2.0,
    seed: int = 0
):
    """
    Generate a sparse linear regression dataset.

    Parameters
    ----------
    n, p, s : int
        Sample size, number of features, sparsity.
    snr : float
        Signal-to-noise ratio.
    seed : int

    Returns
    -------
    X : ndarray (n, p) — standardised
    y : ndarray (n,)   — centred
    beta : ndarray (p,) — true coefficients
    active : ndarray   — indices of active features
    sigma : float      — true noise std
    """
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n, p))
    X = (X - X.mean(0)) / X.std(0)

    # Active features with alternating-sign coefficients
    active = rng.choice(p, size=s, replace=False)
    beta = np.zeros(p)
    beta[active] = rng.uniform(0.5, 2.0, size=s) * rng.choice([-1, 1], size=s)

    signal = X @ beta
    sigma = np.std(signal) / np.sqrt(snr)
    y = signal + rng.standard_normal(n) * sigma
    y = y - y.mean()

    return X, y, beta, active, sigma


# Quick sanity check
X0, y0, beta0, active0, sigma0 = generate_sparse_linear(n=150, p=300, s=10)
print(f"Shape: X={X0.shape}, y={y0.shape}")
print(f"True active features (first 5): {sorted(active0)[:5]}")
print(f"True sigma: {sigma0:.3f}")

In [ ]:
def naive_ols_coverage_on_selected(
    n: int,
    p: int,
    s: int,
    n_reps: int = 500,
    alpha: float = 0.05,
    snr: float = 2.0
) -> dict:
    """
    Monte Carlo coverage study.

    For each repetition:
    1. Generate data with known beta.
    2. Run Lasso to select features.
    3. Run OLS on selected features -> compute 95% CI.
    4. Check if true beta_j is inside the CI for active features that were selected.

    Returns
    -------
    dict with keys:
        'naive_coverage'   : float — actual coverage of stated (1-alpha) CI
        'selection_recall' : float — fraction of true active features selected
        'n_selected_mean'  : float — mean number of selected features
    """
    covered_naive = []
    selection_recalls = []

    for rep in range(n_reps):
        X, y, beta, active, sigma = generate_sparse_linear(n, p, s, snr=snr, seed=rep)

        # Step 1: Lasso selection
        lasso = LassoCV(cv=3, max_iter=5000)
        lasso.fit(X, y)
        selected = np.where(np.abs(lasso.coef_) > 1e-8)[0]

        if len(selected) < 2:
            continue

        # Recall
        recall = len(set(selected) & set(active)) / s
        selection_recalls.append(recall)

        # Step 2: OLS on selected features
        X_sel = X[:, selected]
        ols = LinearRegression(fit_intercept=False)
        ols.fit(X_sel, y)

        # Step 3: Compute naive CIs
        residuals = y - ols.predict(X_sel)
        sigma_hat_sq = np.sum(residuals**2) / (n - len(selected))
        XtX_inv = np.linalg.pinv(X_sel.T @ X_sel)
        se = np.sqrt(sigma_hat_sq * np.diag(XtX_inv))
        z_crit = stats.norm.ppf(1 - alpha / 2)
        ci_lo = ols.coef_ - z_crit * se
        ci_hi = ols.coef_ + z_crit * se

        # Step 4: Check coverage for true active features that were selected
        for j, feat in enumerate(selected):
            if feat in active:  # only assess true active features
                covered = (ci_lo[j] <= beta[feat] <= ci_hi[j])
                covered_naive.append(covered)

    return {
        'naive_coverage': np.mean(covered_naive),
        'selection_recall': np.mean(selection_recalls),
        'n_covered': len(covered_naive),
    }


print("Running coverage simulation (500 reps)...")
coverage_results = naive_ols_coverage_on_selected(n=150, p=300, s=10, n_reps=500)

print(f"\nNaive OLS on Lasso-selected features:")
print(f"  Stated coverage: 95%")
print(f"  Actual coverage: {coverage_results['naive_coverage']:.1%}")
print(f"  Selection recall: {coverage_results['selection_recall']:.1%}")
print(f"  Intervals assessed: {coverage_results['n_covered']}")
print(f"\n  Coverage deficit: {0.95 - coverage_results['naive_coverage']:.1%} below stated level")

## 2. Implementing the Debiased Lasso

The Debiased Lasso corrects the shrinkage bias of the Lasso estimator by adding a correction term based on an approximate precision matrix $\hat{\mathbf{\Theta}} \approx (\mathbf{X}^\top \mathbf{X}/n)^{-1}$.

**Formula:**
$$\hat{\boldsymbol{\beta}}^{\text{d}} = \hat{\boldsymbol{\beta}}^{\text{Lasso}} + \frac{1}{n}\hat{\mathbf{\Theta}}\mathbf{X}^\top(\mathbf{y} - \mathbf{X}\hat{\boldsymbol{\beta}}^{\text{Lasso}})$$

In [ ]:
def nodewise_lasso_column(X: np.ndarray, j: int, mu: float = None) -> np.ndarray:
    """
    Compute column j of the approximate precision matrix via nodewise Lasso.

    Regresses X[:, j] on all other columns using Lasso and returns the
    j-th column of the approximate precision matrix.

    Parameters
    ----------
    X : ndarray (n, p)
    j : int
        Feature index for which to compute the precision column.
    mu : float
        Lasso regularisation for nodewise regression. Defaults to sqrt(log(p)/n).

    Returns
    -------
    theta_j : ndarray (p,)
        j-th column of approximate precision matrix.
    """
    n, p = X.shape
    if mu is None:
        mu = np.sqrt(np.log(p) / n)

    # Remove feature j and regress x_j on the rest
    X_minus_j = np.delete(X, j, axis=1)
    lasso = Lasso(alpha=mu, fit_intercept=False, max_iter=5000)
    lasso.fit(X_minus_j, X[:, j])
    gamma_j = lasso.coef_  # (p-1,) coefficients

    # Residual of x_j not explained by other features
    m_j = X[:, j] - X_minus_j @ gamma_j

    # Normalisation constant
    tau_j = (m_j @ X[:, j]) / n

    # Build column j of Theta_hat
    theta_j = np.zeros(p)
    theta_j[j] = 1.0 / tau_j
    other_idx = np.delete(np.arange(p), j)
    theta_j[other_idx] = -gamma_j / tau_j

    return theta_j


def debiased_lasso(
    X: np.ndarray,
    y: np.ndarray,
    features: np.ndarray,
    alpha_ci: float = 0.05
) -> dict:
    """
    Debiased Lasso confidence intervals (van de Geer et al., 2014).

    Fits Lasso on the full dataset, then debiases the selected features
    using the nodewise Lasso precision matrix estimate.

    Parameters
    ----------
    X : ndarray (n, p)
    y : ndarray (n,)
    features : ndarray
        Subset of feature indices for which to compute debiased CIs.
    alpha_ci : float
        Confidence level (0.05 for 95% CI).

    Returns
    -------
    dict with keys:
        'beta_lasso'    : ndarray — Lasso estimates for features
        'beta_debiased' : ndarray — Debiased estimates for features
        'se'            : ndarray — Standard errors
        'ci_lo', 'ci_hi': ndarray — Confidence interval bounds
        'pvalues'       : ndarray — Two-sided p-values
    """
    n, p = X.shape
    k = len(features)

    # Step 1: Fit Lasso on all features
    lasso = LassoCV(cv=5, max_iter=10000, fit_intercept=False)
    lasso.fit(X, y)
    beta_lasso_full = lasso.coef_  # (p,)

    # Step 2: Compute Lasso residuals
    residuals = y - X @ beta_lasso_full

    # Step 3: Estimate sigma^2 from residuals
    n_selected = np.sum(np.abs(beta_lasso_full) > 1e-8)
    sigma_hat_sq = np.sum(residuals**2) / max(n - n_selected, 1)

    # Step 4: For each feature in 'features', compute debiased estimate
    beta_debiased = np.zeros(k)
    se = np.zeros(k)

    for idx, j in enumerate(features):
        # Nodewise regression for column j of approximate precision matrix
        theta_j = nodewise_lasso_column(X, j)

        # Debiasing correction for feature j
        correction_j = (theta_j @ X.T @ residuals) / n

        beta_debiased[idx] = beta_lasso_full[j] + correction_j

        # Standard error: sigma * sqrt(Theta_jj) / sqrt(n)
        se[idx] = np.sqrt(sigma_hat_sq * theta_j[j]) / np.sqrt(n)

    # Step 5: Confidence intervals and p-values
    z_crit = stats.norm.ppf(1 - alpha_ci / 2)
    ci_lo = beta_debiased - z_crit * se
    ci_hi = beta_debiased + z_crit * se
    z_stats = beta_debiased / (se + 1e-12)
    pvalues = 2 * stats.norm.sf(np.abs(z_stats))

    return {
        'beta_lasso': beta_lasso_full[features],
        'beta_debiased': beta_debiased,
        'se': se,
        'ci_lo': ci_lo,
        'ci_hi': ci_hi,
        'pvalues': pvalues,
    }


# Demonstrate on a single dataset
X_demo, y_demo, beta_demo, active_demo, sigma_demo = generate_sparse_linear(
    n=150, p=100, s=8, snr=3.0, seed=7
)

print("Running debiased Lasso on demo dataset (p=100, n=150)...")
lasso_demo = LassoCV(cv=5, max_iter=5000, fit_intercept=False)
lasso_demo.fit(X_demo, y_demo)
selected_demo = np.where(np.abs(lasso_demo.coef_) > 1e-8)[0]

print(f"Lasso selected {len(selected_demo)} features")

result_debiased = debiased_lasso(X_demo, y_demo, features=selected_demo)

print("\nDebiased Lasso CIs for selected features:")
print(f"{'Feature':>8} {'True β':>8} {'Lasso β':>10} {'Debiased β':>12} {'95% CI':>24} {'Covers?':>8}")
print("-" * 75)
for i, feat in enumerate(selected_demo[:10]):  # show first 10
    true_beta = beta_demo[feat]
    covers = result_debiased['ci_lo'][i] <= true_beta <= result_debiased['ci_hi'][i]
    print(f"{feat:>8} {true_beta:>8.3f} {result_debiased['beta_lasso'][i]:>10.3f} "
          f"{result_debiased['beta_debiased'][i]:>12.3f} "
          f"[{result_debiased['ci_lo'][i]:>6.3f}, {result_debiased['ci_hi'][i]:>6.3f}] "
          f"{'YES' if covers else 'NO':>8}")

## 3. Data Splitting for Valid Inference

Data splitting uses the first half of observations for feature selection and the second half for OLS inference. Because the selection step is independent of the inference data, classical OLS theory applies exactly — no post-selection correction needed.

In [ ]:
def data_split_inference(
    X: np.ndarray,
    y: np.ndarray,
    alpha_ci: float = 0.05,
    split_frac: float = 0.5
) -> dict:
    """
    Data splitting for post-selection inference.

    Uses the first split_frac of data for Lasso selection and the
    remaining (1 - split_frac) for OLS inference.

    Parameters
    ----------
    X : ndarray (n, p)
    y : ndarray (n,)
    alpha_ci : float
        Significance level.
    split_frac : float
        Fraction of data used for selection (remainder for inference).

    Returns
    -------
    dict with keys:
        'selected'      : ndarray — selected feature indices
        'beta_ols'      : ndarray — OLS estimates on inference set
        'se'            : ndarray — standard errors
        'ci_lo', 'ci_hi': ndarray — confidence interval bounds
        'pvalues'       : ndarray — two-sided p-values
        'n_inference'   : int     — inference set size
    """
    n, p = X.shape
    n_sel = int(n * split_frac)

    # Random split
    perm = np.random.permutation(n)
    sel_idx = perm[:n_sel]
    inf_idx = perm[n_sel:]

    X_sel, y_sel = X[sel_idx], y[sel_idx]
    X_inf, y_inf = X[inf_idx], y[inf_idx]

    # Step 1: Selection on selection set
    lasso = LassoCV(cv=3, max_iter=5000, fit_intercept=False)
    lasso.fit(X_sel, y_sel)
    selected = np.where(np.abs(lasso.coef_) > 1e-8)[0]

    if len(selected) == 0:
        return {'selected': selected, 'beta_ols': np.array([]),
                'se': np.array([]), 'ci_lo': np.array([]),
                'ci_hi': np.array([]), 'pvalues': np.array([]),
                'n_inference': len(inf_idx)}

    # Step 2: OLS on inference set — standard CIs are valid here
    X_inf_sel = X_inf[:, selected]
    n_inf = len(inf_idx)
    k_sel = len(selected)

    ols = LinearRegression(fit_intercept=False)
    ols.fit(X_inf_sel, y_inf)

    residuals = y_inf - ols.predict(X_inf_sel)
    sigma_sq = np.sum(residuals**2) / (n_inf - k_sel)
    XtX_inv = np.linalg.pinv(X_inf_sel.T @ X_inf_sel)
    se = np.sqrt(sigma_sq * np.diag(XtX_inv))

    # t-distribution with (n_inf - k_sel) degrees of freedom
    t_crit = stats.t.ppf(1 - alpha_ci / 2, df=n_inf - k_sel)
    ci_lo = ols.coef_ - t_crit * se
    ci_hi = ols.coef_ + t_crit * se
    t_stats = ols.coef_ / (se + 1e-12)
    pvalues = 2 * stats.t.sf(np.abs(t_stats), df=n_inf - k_sel)

    return {
        'selected': selected,
        'beta_ols': ols.coef_,
        'se': se,
        'ci_lo': ci_lo,
        'ci_hi': ci_hi,
        'pvalues': pvalues,
        'n_inference': n_inf,
    }


# Demo on the same dataset
np.random.seed(0)
ds_result = data_split_inference(X_demo, y_demo)
print(f"Data splitting — selected {len(ds_result['selected'])} features")
print(f"Inference set size: {ds_result['n_inference']}")

# Coverage check
covered_ds = []
for i, feat in enumerate(ds_result['selected']):
    if feat in set(active_demo):
        covers = ds_result['ci_lo'][i] <= beta_demo[feat] <= ds_result['ci_hi'][i]
        covered_ds.append(covers)

if covered_ds:
    print(f"Coverage of true active features (data split): {np.mean(covered_ds):.1%}")

## 4. Coverage Comparison: Naive vs Debiased vs Data Splitting

We run a systematic Monte Carlo study comparing coverage rates across all three methods.

In [ ]:
def coverage_study(
    n: int = 150,
    p: int = 100,
    s: int = 8,
    n_reps: int = 200,
    alpha_ci: float = 0.05,
    snr: float = 3.0
) -> dict:
    """
    Compare coverage of naive, debiased, and data-split CIs.

    For each method, reports the fraction of intervals that contain
    the true coefficient value for selected active features.
    """
    covered_naive = []
    covered_debiased = []
    covered_split = []
    ci_widths_debiased = []
    ci_widths_split = []

    for rep in range(n_reps):
        X, y, beta, active, sigma = generate_sparse_linear(n, p, s, snr=snr, seed=rep)

        # --- Naive method ---
        lasso = LassoCV(cv=3, max_iter=5000, fit_intercept=False)
        lasso.fit(X, y)
        selected = np.where(np.abs(lasso.coef_) > 1e-8)[0]

        if len(selected) >= 2:
            X_sel = X[:, selected]
            ols_naive = LinearRegression(fit_intercept=False).fit(X_sel, y)
            resid_naive = y - ols_naive.predict(X_sel)
            sigma_naive = np.sum(resid_naive**2) / max(n - len(selected), 1)
            XtX_inv = np.linalg.pinv(X_sel.T @ X_sel)
            se_naive = np.sqrt(sigma_naive * np.diag(XtX_inv))
            z_c = stats.norm.ppf(1 - alpha_ci / 2)

            for j, feat in enumerate(selected):
                if feat in active:
                    ci_lo = ols_naive.coef_[j] - z_c * se_naive[j]
                    ci_hi = ols_naive.coef_[j] + z_c * se_naive[j]
                    covered_naive.append(ci_lo <= beta[feat] <= ci_hi)

        # --- Debiased Lasso ---
        if len(selected) >= 1:
            try:
                db = debiased_lasso(X, y, features=selected, alpha_ci=alpha_ci)
                for j, feat in enumerate(selected):
                    if feat in active:
                        covered_debiased.append(db['ci_lo'][j] <= beta[feat] <= db['ci_hi'][j])
                        ci_widths_debiased.append(db['ci_hi'][j] - db['ci_lo'][j])
            except Exception:
                pass  # skip if nodewise regression fails (rare numerical issues)

        # --- Data splitting ---
        ds = data_split_inference(X, y, alpha_ci=alpha_ci)
        for j, feat in enumerate(ds['selected']):
            if feat in active:
                covered_split.append(ds['ci_lo'][j] <= beta[feat] <= ds['ci_hi'][j])
                ci_widths_split.append(ds['ci_hi'][j] - ds['ci_lo'][j])

    return {
        'naive': np.mean(covered_naive) if covered_naive else np.nan,
        'debiased': np.mean(covered_debiased) if covered_debiased else np.nan,
        'split': np.mean(covered_split) if covered_split else np.nan,
        'width_debiased': np.mean(ci_widths_debiased) if ci_widths_debiased else np.nan,
        'width_split': np.mean(ci_widths_split) if ci_widths_split else np.nan,
        'n_naive': len(covered_naive),
        'n_debiased': len(covered_debiased),
        'n_split': len(covered_split),
    }


print("Running coverage study (200 reps, p=100, n=150, s=8)...")
cov_results = coverage_study(n=150, p=100, s=8, n_reps=200, snr=3.0)

print("\nCoverage Study Results:")
print("=" * 55)
print(f"{'Method':<20} {'Coverage':>10} {'Mean CI width':>15} {'n intervals':>12}")
print("-" * 55)
print(f"{'Naive OLS':<20} {cov_results['naive']:>10.1%} {'N/A':>15} {cov_results['n_naive']:>12}")
print(f"{'Debiased Lasso':<20} {cov_results['debiased']:>10.1%} {cov_results['width_debiased']:>15.4f} {cov_results['n_debiased']:>12}")
print(f"{'Data splitting':<20} {cov_results['split']:>10.1%} {cov_results['width_split']:>15.4f} {cov_results['n_split']:>12}")
print(f"\nTarget coverage: 95%")

## 5. Visualising Coverage and CI Width

A visual comparison of the three approaches, showing coverage rates and the width-coverage trade-off.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

methods = ['Naive OLS\n(on selected)', 'Debiased\nLasso', 'Data\nSplitting']
coverages = [
    cov_results['naive'],
    cov_results['debiased'],
    cov_results['split']
]
colors = ['#e63333', '#2196F3', '#4CAF50']

# Panel 1: Coverage bar chart
ax = axes[0]
bars = ax.bar(methods, [100 * c for c in coverages], color=colors, alpha=0.8, width=0.5)
ax.axhline(95, color='black', linestyle='--', linewidth=2, label='Target 95%')

# Label bars with values
for bar, cov in zip(bars, coverages):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{100*cov:.1f}%', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Actual Coverage (%)')
ax.set_title('Coverage of "95%" Confidence Intervals')
ax.set_ylim(0, 102)
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Panel 2: CI width comparison (debiased vs split)
ax = axes[1]
width_methods = ['Debiased Lasso', 'Data Splitting']
widths = [cov_results['width_debiased'], cov_results['width_split']]
ax.bar(width_methods, widths, color=['#2196F3', '#4CAF50'], alpha=0.8, width=0.4)

# Annotate with values
for i, (method, width) in enumerate(zip(width_methods, widths)):
    ax.text(i, width + 0.001, f'{width:.4f}', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Mean CI Width')
ax.set_title('Average Width of 95% CI\n(narrower = more efficient)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.suptitle(
    f'Post-Selection Inference Comparison (n={150}, p={100}, s={8}, 200 reps)',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print(f"  Naive OLS: {100*coverages[0]:.1f}% vs stated 95% — severely anti-conservative")
print(f"  Debiased Lasso: {100*coverages[1]:.1f}% — approximately correct")
print(f"  Data splitting: {100*coverages[2]:.1f}% — conservative (uses half the data for inference)")
print(f"  Width ratio (debiased/split): {cov_results['width_debiased']/cov_results['width_split']:.2f}x")

## 6. The Complete Pipeline: Screen → Select → Infer

We now assemble the complete Module 8 workflow on a realistic high-dimensional dataset:
1. **Screen** with SIS (Guide 01) to reduce $p = 3{,}000$ to $d_n \sim 45$
2. **Select** with Lasso on the screened features
3. **Infer** with the Debiased Lasso on the selected features

In [ ]:
def sis_screen(X: np.ndarray, y: np.ndarray, d_n: int = None) -> tuple:
    """SIS screening — see Notebook 01 for full implementation."""
    n, p = X.shape
    if d_n is None:
        d_n = int(n / np.log(n))
    y_ctr = y - y.mean()
    omega = np.abs(X.T @ y_ctr) / n
    screened_idx = np.argsort(omega)[::-1][:d_n]
    return screened_idx, omega


def complete_pipeline(
    X: np.ndarray,
    y: np.ndarray,
    true_active: np.ndarray,
    beta_true: np.ndarray,
    alpha_ci: float = 0.05
) -> dict:
    """
    Full Module 8 pipeline: SIS → Lasso → Debiased Lasso inference.

    Parameters
    ----------
    X : ndarray (n, p)
    y : ndarray (n,)
    true_active : ndarray — ground truth active features (for evaluation only)
    beta_true : ndarray (p,) — true coefficients (for evaluation only)
    alpha_ci : float — confidence level

    Returns
    -------
    dict with pipeline outputs and evaluation metrics
    """
    n, p = X.shape
    results = {}

    # ===== STEP 1: Screen =====
    d_n = int(n / np.log(n))
    screened_idx, omega = sis_screen(X, y, d_n=d_n)
    X_screened = X[:, screened_idx]

    # Evaluate screening
    screen_recall = len(set(screened_idx) & set(true_active)) / len(true_active)
    results['screening'] = {
        'n_screened': d_n,
        'recall': screen_recall,
        'screened_idx': screened_idx
    }
    print(f"Step 1 — SIS screening: p={p} → {d_n} features, recall={screen_recall:.0%}")

    # ===== STEP 2: Select =====
    lasso_step2 = LassoCV(cv=5, max_iter=10000, fit_intercept=False)
    lasso_step2.fit(X_screened, y)

    active_in_screened = np.where(np.abs(lasso_step2.coef_) > 1e-8)[0]
    selected_original = screened_idx[active_in_screened]  # back to original indices

    sel_tp = len(set(selected_original) & set(true_active))
    sel_fp = len(set(selected_original) - set(true_active))
    results['selection'] = {
        'selected': selected_original,
        'n_selected': len(selected_original),
        'tp': sel_tp, 'fp': sel_fp,
        'precision': sel_tp / max(len(selected_original), 1),
        'recall': sel_tp / len(true_active)
    }
    print(f"Step 2 — Lasso: {len(selected_original)} features selected, "
          f"TP={sel_tp}, FP={sel_fp}")

    # ===== STEP 3: Infer =====
    if len(selected_original) >= 1:
        db = debiased_lasso(X, y, features=selected_original, alpha_ci=alpha_ci)

        # Evaluate coverage on true active features that were selected
        covered = []
        for j, feat in enumerate(selected_original):
            if feat in set(true_active):
                covers = db['ci_lo'][j] <= beta_true[feat] <= db['ci_hi'][j]
                covered.append(covers)

        results['inference'] = {
            'method': 'Debiased Lasso',
            'selected': selected_original,
            'beta_debiased': db['beta_debiased'],
            'ci_lo': db['ci_lo'],
            'ci_hi': db['ci_hi'],
            'pvalues': db['pvalues'],
            'coverage_of_selected_active': np.mean(covered) if covered else np.nan,
            'n_covered': len(covered)
        }
        print(f"Step 3 — Debiased Lasso inference: {alpha_ci:.0%} CI coverage on "
              f"selected active features = {np.mean(covered) if covered else 'N/A':.0%}")
    else:
        results['inference'] = {'selected': np.array([])}
        print("Step 3 — No features selected; skipping inference")

    return results


# Generate a high-dimensional dataset for the complete pipeline
print("Generating high-dimensional dataset (n=200, p=3000, s=10)...")
X_full, y_full, beta_full, active_full, sigma_full = generate_sparse_linear(
    n=200, p=3000, s=10, snr=3.0, seed=99
)

print("\nRunning complete pipeline...")
print("=" * 55)
pipeline_results = complete_pipeline(
    X_full, y_full, active_full, beta_full, alpha_ci=0.05
)

## 7. Visualising the Final Inference Results

Forest plot of debiased Lasso confidence intervals for selected features, with true values marked.

In [ ]:
inf = pipeline_results['inference']
if len(inf.get('selected', [])) > 0:
    selected = inf['selected']
    beta_d = inf['beta_debiased']
    ci_lo = inf['ci_lo']
    ci_hi = inf['ci_hi']
    true_betas = beta_full[selected]

    # Sort by estimated effect size
    sort_order = np.argsort(np.abs(beta_d))[::-1]

    fig, ax = plt.subplots(figsize=(9, max(5, len(selected) * 0.5 + 2)))

    y_positions = np.arange(len(selected))

    for i, idx in enumerate(sort_order):
        feat = selected[idx]
        is_active = feat in set(active_full)
        covers = ci_lo[idx] <= true_betas[idx] <= ci_hi[idx]

        # Horizontal CI
        color = '#2196F3' if (is_active and covers) else \
                '#e63333' if (is_active and not covers) else \
                '#9E9E9E'

        ax.plot([ci_lo[idx], ci_hi[idx]], [i, i], color=color, linewidth=2, alpha=0.8)
        ax.scatter(beta_d[idx], i, color=color, s=60, zorder=5)

        # True value marker (diamond)
        if is_active:
            ax.scatter(true_betas[idx], i, color='black', s=80,
                       marker='D', zorder=6, label='True β' if i == 0 else '')

    ax.axvline(0, color='grey', linestyle='--', alpha=0.5)
    ax.set_yticks(range(len(selected)))
    ax.set_yticklabels(
        [f"Feature {selected[sort_order[i]]} {'(active)' if selected[sort_order[i]] in active_full else '(noise)'}" 
         for i in range(len(selected))],
        fontsize=9
    )
    ax.set_xlabel('Coefficient value')
    ax.set_title('Debiased Lasso 95% Confidence Intervals\n(blue=TP covered, red=TP missed, grey=FP)')

    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='#2196F3', linewidth=2, label='True active — CI covers'),
        Line2D([0], [0], color='#e63333', linewidth=2, label='True active — CI misses'),
        Line2D([0], [0], color='#9E9E9E', linewidth=2, label='False positive'),
        Line2D([0], [0], color='black', linewidth=0, marker='D', markersize=8,
               label='True coefficient value'),
    ]
    ax.legend(handles=legend_elements, fontsize=9, loc='lower right')
    plt.tight_layout()
    plt.show()
else:
    print("No features selected — nothing to plot.")

## Summary

### Key Takeaways

1. **Naive OLS on selected features is anti-conservative**: In the simulation, a stated 95% CI achieves actual coverage of around 50–60%. This is not a rare edge case — it is the systematic consequence of using the same data for selection and inference.

2. **Debiased Lasso corrects the bias**: The correction $\frac{1}{n}\hat{\mathbf{\Theta}}\mathbf{X}^\top(\mathbf{y} - \mathbf{X}\hat{\boldsymbol{\beta}}^{\text{Lasso}})$ amplifies the Lasso residuals through the approximate precision matrix, removing the shrinkage bias. The result is asymptotically valid CIs.

3. **Data splitting is simpler but costs power**: CIs are wider by a factor of $\sqrt{2}$ because only half the data is used for inference, but they are valid without any asymptotic argument.

4. **The complete pipeline**: SIS (screening, $O(pn)$) → Lasso (selection, $O(d_n^2 n)$) → Debiased Lasso (inference, $O(d_n^2 n)$) handles ultra-high dimensional data efficiently while producing valid inferential statements.

### What's Next

Module 9 (`module_09_causal`) extends feature selection to causal inference: which features have a *causal* effect on the outcome, beyond mere statistical association.

### Additional Resources

- **van de Geer et al. (2014)**: Debiased Lasso — *Annals of Statistics* 42(3), 1166–1202
- **Zhang & Zhang (2014)**: Independent debiasing derivation — *JRSS-B* 76(1), 217–242
- **Lee et al. (2016)**: Selective inference (conditional approach) — *Annals of Statistics* 44(3), 907–927
- `statsmodels.regression.linear_model`: OLS inference tools for the data-splitting step